# 🔧 LoanGuard — Veri Ön-İşleme (Preprocessing)

**Amaç:** Ham veriyi temizleyip encode ederek `02b_feature_engineering.ipynb` için hazırlamak.

**EDA'dan Gelen Bulgular:**
- ✅ Eksik değer yok → Imputation gerekmez
- ✅ Tekrar eden satır yok
- ⚠️ Sınıf dengesizliği (~7.6:1) → `scale_pos_weight` modelde uygulanacak

**Bu Notebook'ta Yapılacaklar:**
1. YAML konfigürasyonunu yükle
2. ID sütununu kaldır
3. Kategorik değişkenleri encode et (LabelEncoder)
4. Temizlenmiş veriyi `data/interim/loans_cleaned.csv` olarak kaydet

> **Not:** Train/Test Split ve Scaling işlemleri `02b_feature_engineering.ipynb`'de yapılır.
>

---

## 1. Kütüphaneler & Konfigürasyon

Proje konfigürasyonu `configs/model_params.yaml` dosyasından okunur.  
Bu sayede parametreler tek bir merkezden yönetilir; notebook'a sabit değer yazılmaz.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
import joblib
import warnings
import os

from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# YAML konfigürasyon dosyasını yükle
# ---------------------------------------------------------------------------
with open("../configs/model_params.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

ENCODER_TYPE    = config["preprocessing"]["categorical_encoder"]

print("✅ Konfigürasyon yüklendi:")
print(f"   Encoder         : {ENCODER_TYPE}")


✅ Konfigürasyon yüklendi:
   Encoder         : LabelEncoder


In [2]:
# ---------------------------------------------------------------------------
# Sütun gruplarını tanımla
# ---------------------------------------------------------------------------
NUMERICAL_COLS = [
    "Age", "Income", "LoanAmount", "CreditScore",
    "MonthsEmployed", "NumCreditLines", "InterestRate",
    "LoanTerm", "DTIRatio"
]

CATEGORICAL_COLS = [
    "Education", "EmploymentType", "MaritalStatus",
    "HasMortgage", "HasDependents", "LoanPurpose", "HasCoSigner"
]

TARGET_COL = "Default"
ID_COL     = "LoanID"

# Çıktı klasörleri
INTERIM_DIR = "../data/interim/"   # Temizlenmiş ama bölünmemiş/ölçeklenmemiş ara veri
MODELS_DIR  = "../models/"
FIGURES_DIR = "../reports/figures/"
os.makedirs(INTERIM_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams.update({"figure.figsize": (14, 6), "figure.dpi": 120,
                     "axes.titlesize": 14, "axes.titleweight": "bold"})

print("✅ Sabitler ve çıktı klasörleri hazır.")

✅ Sabitler ve çıktı klasörleri hazır.


---
## 2. Veri Yükleme & ID Sütununu Kaldırma

`LoanID` sütunu sadece bir kimlik numarasıdır; model için bilgi taşımaz.  
Modele verilirse hem gereksiz gürültü oluşturur hem de boyut artar.

In [3]:
df = pd.read_csv("../data/raw/Loan_default.csv")
print(f"📐 Ham veri boyutu: {df.shape[0]:,} satır × {df.shape[1]} sütun")

df = df.drop(columns=[ID_COL])
print(f"🗑️  '{ID_COL}' sütunu kaldırıldı → Yeni boyut: {df.shape[0]:,} × {df.shape[1]}")
df.head()

📐 Ham veri boyutu: 255,347 satır × 18 sütun
🗑️  'LoanID' sütunu kaldırıldı → Yeni boyut: 255,347 × 17


,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default
0,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0
1,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0
2,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1
3,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0
4,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0


---
## 4. Kategorik Değişkenleri Encode Etme

ML modelleri sayısal verilerle çalışır; metin (string) değerleri anlayamaz.  
`LabelEncoder` kullanıldı — XGBoost ağaç tabanlıdır, sıralamayı kendisi öğrenir.

**Önemli:** Her sütun için ayrı bir `LabelEncoder` nesnesi oluşturulur ve kaydedilir.  
API'de yeni veri geldiğinde aynı dönüşüm uygulanabilir.

In [4]:
label_encoders = {}

print("🔄 Kategorik sütunlar encode ediliyor...\n")
print(f"{'Sütun':<18} {'Benzersiz Değerler':<40} {'Eşleşme (Mapping)'}")
print("-" * 90)

for col in CATEGORICAL_COLS:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le
    mapping = dict(zip(le.classes_, le.transform(le.classes_)))
    print(f"{col:<18} {str(le.classes_):<40} {mapping}")

print(f"\n✅ {len(CATEGORICAL_COLS)} kategorik sütun encode edildi.")

🔄 Kategorik sütunlar encode ediliyor...

Sütun              Benzersiz Değerler                       Eşleşme (Mapping)
------------------------------------------------------------------------------------------
Education          ["Bachelor's" 'High School' "Master's" 'PhD'] {"Bachelor's": 0, 'High School': 1, "Master's": 2, 'PhD': 3}
EmploymentType     ['Full-time' 'Part-time' 'Self-employed' 'Unemployed'] {'Full-time': 0, 'Part-time': 1, 'Self-employed': 2, 'Unemployed': 3}
MaritalStatus      ['Divorced' 'Married' 'Single']          {'Divorced': 0, 'Married': 1, 'Single': 2}
HasMortgage        ['No' 'Yes']                             {'No': 0, 'Yes': 1}
HasDependents      ['No' 'Yes']                             {'No': 0, 'Yes': 1}
LoanPurpose        ['Auto' 'Business' 'Education' 'Home' 'Other'] {'Auto': 0, 'Business': 1, 'Education': 2, 'Home': 3, 'Other': 4}
HasCoSigner        ['No' 'Yes']                             {'No': 0, 'Yes': 1}

✅ 7 kategorik sütun encode edildi.


In [5]:
print("📋 Encode sonrası veri tipleri:\n")
print(df.dtypes.to_string())
print(f"\n✅ Tüm sütunlar sayısal: {df.select_dtypes(include='object').shape[1] == 0}")

📋 Encode sonrası veri tipleri:

Age                 int64
Income              int64
LoanAmount          int64
CreditScore         int64
MonthsEmployed      int64
NumCreditLines      int64
InterestRate      float64
LoanTerm            int64
DTIRatio          float64
Education           int32
EmploymentType      int32
MaritalStatus       int32
HasMortgage         int32
HasDependents       int32
LoanPurpose         int32
HasCoSigner         int32
Default             int64

✅ Tüm sütunlar sayısal: True


---
## 5. Temizlenmiş Veriyi Kaydetme

Train/Test split ve scaling **bu notebook'ta yapılmaz** — `02b_feature_engineering.ipynb` sorumluluğundadır.  
Bu adımda sadece temizlenmiş veri kaydedilir.

| Dosya | İçerik | Kullanım Yeri |
|-------|--------|---------------|
| `data/interim/loans_cleaned.csv` | ID drop + encode edilmiş tam veri (split yok) | Notebook 02b |
| `models/label_encoders.joblib` | LabelEncoder nesneleri | API — yeni veri dönüştürme |

In [6]:
# ---------------------------------------------------------------------------
# Temizlenmiş veriyi kaydet (train/test split ve scaling YAPILMADI)
# ---------------------------------------------------------------------------
cleaned_path = f"{INTERIM_DIR}loans_cleaned.csv"
df.to_csv(cleaned_path, index=False)

# LabelEncoder nesnelerini kaydet (API inference için)
joblib.dump(label_encoders, f"{MODELS_DIR}label_encoders.joblib")

print("✅ Dosyalar kaydedildi:")
print(f"   📄 loans_cleaned.csv     → {os.path.getsize(cleaned_path) / 1024:.1f} KB")
print(f"   📦 label_encoders.joblib → {MODELS_DIR}")
print(f"\n📐 Veri boyutu   : {df.shape[0]:,} satır × {df.shape[1]} sütun")
print(f"📌 Sütunlar      : {list(df.columns)}")

✅ Dosyalar kaydedildi:
   📄 loans_cleaned.csv     → 13839.2 KB
   📦 label_encoders.joblib → ../models/

📐 Veri boyutu   : 255,347 satır × 17 sütun
📌 Sütunlar      : ['Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed', 'NumCreditLines', 'InterestRate', 'LoanTerm', 'DTIRatio', 'Education', 'EmploymentType', 'MaritalStatus', 'HasMortgage', 'HasDependents', 'LoanPurpose', 'HasCoSigner', 'Default']


In [7]:
# ---------------------------------------------------------------------------
# Doğrulama: Kaydedilen veriyi yeniden yükle
# ---------------------------------------------------------------------------
df_check = pd.read_csv(cleaned_path)
le_check  = joblib.load(f"{MODELS_DIR}label_encoders.joblib")

print("🔍 Doğrulama Sonuçları:")
print(f"   Boyut          : {df_check.shape} → {'✅' if df_check.shape == df.shape else '❌'}")
print(f"   Eksik değer    : {df_check.isnull().sum().sum()} → {'✅' if df_check.isnull().sum().sum() == 0 else '❌'}")
print(f"   String sütun   : {df_check.select_dtypes('object').shape[1]} → {'✅' if df_check.select_dtypes('object').shape[1] == 0 else '❌'}")
print(f"   Encoder sayısı : {len(le_check)} → {'✅' if len(le_check) == len(CATEGORICAL_COLS) else '❌'}")
print(f"\n⚖️  Hedef dağılımı:")
vc = df_check[TARGET_COL].value_counts()
print(f"   Ödedi (0)    : {vc.get(0, 0):,} (%{vc.get(0, 0) / len(df_check) * 100:.1f})")
print(f"   Temerrüt (1) : {vc.get(1, 0):,} (%{vc.get(1, 0) / len(df_check) * 100:.1f})")

🔍 Doğrulama Sonuçları:
   Boyut          : (255347, 17) → ✅
   Eksik değer    : 0 → ✅
   String sütun   : 0 → ✅
   Encoder sayısı : 7 → ✅

⚖️  Hedef dağılımı:
   Ödedi (0)    : 225,694 (%88.4)
   Temerrüt (1) : 29,653 (%11.6)


---
## 📋 Özet

| # | İşlem | Detay |
|---|-------|-------|
| 1 | ID kaldırma | `LoanID` sütunu düşürüldü |
| 2 | Encoding | 7 kategorik sütun → LabelEncoder |
| 3 | Kaydetme | `data/interim/loans_cleaned.csv` + `label_encoders.joblib` |

### Sonraki Notebook (`02b_feature_engineering.ipynb`)
- Finansal rasyolar ekle: `LoanToIncome`, `PaymentToIncome`, `CreditAgePerLine`, `TotalDebtBurden`
- Train/Test split (%80/%20 stratified)
- StandardScaler uygula (yeni özellikler dahil 13 sayısal sütun)
- `data/processed/` klasörüne kaydet → Notebook 03'ün okuyacağı veriler

---
*Ön-işleme tamamlandı. Temizlenmiş veri `data/interim/` klasörüne kaydedilmiştir.*